### **Data Quality Audit**

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(
        ["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"],
        20
    ),
    "onboarded": pd.date_range(
        "2024-01-01",
        periods=20,
        freq="MS"
    ).strftime("%Y-%m-%d"),
})

txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(
        ["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"],
        n
    ),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (
        pd.to_datetime("2026-03-01")
        + pd.to_timedelta(rng.integers(0, 60, n), unit="D")
    ).strftime("%Y-%m-%d"),
})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat(
    [txns, txns.sample(12, random_state=1)],
    ignore_index=True
)

txns.loc[
    rng.choice(len(txns), 8, replace=False),
    "agent_id"
] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

print("Files created successfully")

Files created successfully


In [ ]:
import pandas as pd

txns = pd.read_csv(
    "agent_transactions.csv"
)

agents = pd.read_csv(
    "agents.csv"
)

In [ ]:
audit = pd.DataFrame({
    "issue": [
        "Duplicate transactions",
        "Missing amounts",
        "Invalid amounts",
        "Unknown agents",
        "Inconsistent transaction types",
        "Inconsistent date formats"
    ],
    "dimension": [
        "Uniqueness",
        "Completeness",
        "Validity",
        "Accuracy",
        "Consistency",
        "Consistency"
    ],
    "column": [
        "txn_id",
        "amount",
        "amount",
        "agent_id",
        "txn_type",
        "txn_date"
    ],
    "rows_affected": [
        txns.duplicated().sum(),
        txns["amount"].isna().sum(),
        (txns["amount"]=="N/A").sum(),
        (txns["agent_id"]=="AG099").sum(),
        txns["txn_type"].nunique(),
        txns["txn_date"].str.contains("/").sum()
    ]
})

audit

,issue,dimension,column,rows_affected
0,Duplicate transactions,Uniqueness,txn_id,11
1,Missing amounts,Completeness,amount,22
2,Invalid amounts,Validity,amount,0
3,Unknown agents,Accuracy,agent_id,8
4,Inconsistent transaction types,Consistency,txn_type,5
5,Inconsistent date formats,Consistency,txn_date,5


In [ ]:
clean_txns = (
    txns
    .drop_duplicates()
    .copy()
)
duplicates_removed = (
    len(txns)
    -
    len(clean_txns)
)

print(
    duplicates_removed
)

11


### **Standardise txn_type**

In [ ]:
clean_txns["txn_type"] = (
    clean_txns["txn_type"]
    .str.strip()
    .str.title()
)
clean_txns[
    "txn_type"
].value_counts()

,count
txn_type,
Deposit,374
Withdrawal,227


In [ ]:
clean_txns["amount"] = pd.to_numeric(
    clean_txns["amount"],
    errors="coerce"
)
clean_txns["txn_date"] = pd.to_datetime(
    clean_txns["txn_date"]
    .str.replace(
        "/",
        "-"
    )
)

### **Missing values**

In [ ]:
clean_txns[
    "amount_imputed"
] = (
    clean_txns["amount"]
    .isna()
)
median_amount = (
    clean_txns["amount"]
    .median()
)

clean_txns["amount"] = (
    clean_txns["amount"]
    .fillna(
        median_amount
    )
)


Missing values were imputed using the median because transaction amounts are highly skewed and the median is less affected by extreme values than the mean.

### **Outliers**


In [ ]:
q1 = clean_txns[
    "amount"
].quantile(0.25)

q3 = clean_txns[
    "amount"
].quantile(0.75)

iqr = q3 - q1

upper = q3 + 1.5 * iqr

clean_txns[
    "is_outlier"
] = (
    clean_txns["amount"]
    >
    upper
)
clean_txns[
    clean_txns[
        "amount"
    ] > 200000
]

,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier
306,TX00307,AG012,Deposit,250000.0,2026-03-02,False,True
315,TX00316,AG014,Deposit,250000.0,2026-04-01,False,True
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,False,True
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,False,True
594,TX00595,AG016,Deposit,250000.0,2026-03-11,False,True


The KES 250,000 transactions were flagged as outliers using the IQR rule. These values should be reviewed with business stakeholders before removal because they may represent legitimate high-value transactions.

### **Merge**

In [ ]:
merged = clean_txns.merge(
    agents,
    on="agent_id",
    how="left",
    indicator=True
)
unknown_agents = (
    merged[
        "_merge"
    ] == "left_only"
).sum()

print(
    unknown_agents
)

8


### **County summary**


In [ ]:
county_summary = (
    merged.groupby(
        ["county","txn_type"]
    )
    .agg(
        total_value=("amount","sum"),
        transactions=("txn_id","count"),
        average_value=("amount","mean")
    )
)

county_summary

total_value  transactions  average_value
county  txn_type                                            
Kiambu  Deposit        334737.0            43    7784.581395
        Withdrawal      77052.0            27    2853.777778
Kisumu  Deposit        436425.0            74    5897.635135
        Withdrawal      76376.0            37    2064.216216
Mombasa Deposit        148992.0            72    2069.333333
        Withdrawal     368592.0            61    6042.491803
Nairobi Deposit        543596.0           118    4606.745763
        Withdrawal     162042.0            62    2613.580645
Nakuru  Deposit        184893.0            60    3081.550000
        Withdrawal     328596.0            39    8425.538462

In [ ]:
merged["week"] = (
    merged["txn_date"]
    .dt.isocalendar()
    .week
)

weekly_pivot = pd.pivot_table(
    merged,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

weekly_pivot

week,9,10,11,12,13,14,15,16,17,18
county,,,,,,,,,,
Kiambu,0.0,18317.0,267773.0,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28128.0,27077.0,41077.0,15341.0,29419.0,37439.0,21308.0
Mombasa,6759.0,285671.0,30479.0,25146.0,17050.0,38344.0,47094.0,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52862.0,43109.0,41375.0,282931.0,56788.0,56434.0,65190.0,30936.0
Nakuru,2745.0,56574.0,28730.0,39795.0,19000.0,18145.0,276609.0,26154.0,30862.0,14875.0
